# 04 — Train Decision Transformer (softmax) + Inference Movie

Loads `data/yoked/dataset/actions_synthetic_pretrial.parquet`, encodes each step via `GridCellEncoder` (60-D pose), slices into `(rtg, state, action)` context windows, and trains a softmax-attention `DecisionTransformer` with AdamW + cross-entropy on the rat's action labels.

RTG conditioning uses the canonical `actions_to_reward` column (integer steps until the next reward in the session), cast to float and fed to `embed_rtg`. The DT learns: *given recent context and "reward in N steps," what action did the rat take?* At rollout time, sampling at low RTG_TARGET asks the model to imitate trajectories that lead to a reward soon.

**Hyperparams** mirror notebooks 05 / 06 / 07 except for the architecture knobs, so the only meaningful variable across the four notebooks is the model itself.

**Architecture**: softmax DT (`DecisionTransformer`), `embed_dim=60, num_heads=4 → d_head=15`. State and embed dims are matched (the baseline DT hardcodes them). See notebook 06 for the decoupled variant that lifts that constraint.

## 1. Setup

In [ ]:
from pathlib import Path
import json, time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

import sys; sys.path.insert(0, '../scripts')
import train_dt as T  # reuse the script's windowing helpers

from corner_maze_rl.encoders.grid_cells import GridCellEncoder
from corner_maze_rl.env.corner_maze_env import CornerMazeEnv
from corner_maze_rl.models.decision_transformer import DTConfig, DecisionTransformer

DATA_PATH    = Path('../data/yoked/dataset/actions_synthetic_pretrial.parquet')
RUN_DIR      = Path('../runs/dt/nb04'); RUN_DIR.mkdir(parents=True, exist_ok=True)

CONTEXT_SIZE = 32
EMBED_DIM    = 60       # must equal GridCellEncoder.output_dim for this arch
NUM_HEADS    = 4        # d_head = 60/4 = 15
NUM_LAYERS   = 2
POS_ENCODING = 'learned'
LR           = 1e-3
WEIGHT_DECAY = 1e-4
BATCH_SIZE   = 64
EPOCHS       = 50
VAL_FRAC     = 0.10
SEED         = 0
MAX_SESSIONS = None     # None = all; set e.g. 50 for a smoke run

device = ('cuda' if torch.cuda.is_available()
          else 'mps' if torch.backends.mps.is_available()
          else 'cpu')
torch.manual_seed(SEED); np.random.seed(SEED)
print(f'device={device}  d_head={EMBED_DIM // NUM_HEADS}')

## 2. Load + window the yoked dataset

In [ ]:
df = pd.read_parquet(DATA_PATH)
if MAX_SESSIONS is not None:
    keep = sorted(df['session_id'].unique())[:MAX_SESSIONS]
    df = df[df['session_id'].isin(keep)].reset_index(drop=True)
print(f'rows={len(df):,}  sessions={df["session_id"].nunique()}')

encoder = GridCellEncoder()
assert encoder.output_dim == EMBED_DIM, \
    f'encoder.output_dim={encoder.output_dim} must equal EMBED_DIM={EMBED_DIM}'

full = T.build_dt_dataset(df, encoder, context_size=CONTEXT_SIZE)
n = len(full); n_val = max(1, int(n * VAL_FRAC)); n_train = n - n_val
train_ds, val_ds = torch.utils.data.random_split(
    full, [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED),
)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
print(f'windows: train={n_train:,}  val={n_val:,}')

## 3. Train

In [ ]:
cfg = DTConfig(
    embed_dim=EMBED_DIM, num_actions=5, context_size=CONTEXT_SIZE,
    num_heads=NUM_HEADS, num_layers=NUM_LAYERS, pos_encoding=POS_ENCODING,
)
model = DecisionTransformer(cfg).to(device)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
criterion = nn.CrossEntropyLoss()
n_params = sum(p.numel() for p in model.parameters())
print(f'params={n_params:,}  pos_encoding={POS_ENCODING}')

history = []
t0 = time.time()
for epoch in range(1, EPOCHS + 1):
    model.train()
    sum_loss = sum_correct = sum_tokens = 0
    for rtg, state, action, pos in train_loader:
        rtg = rtg.to(device); state = state.to(device); action = action.to(device)
        pos = pos.to(device) if POS_ENCODING == 'spatial' else None
        logits = model(rtg, state, action, pos_vecs=pos)
        targets = action.argmax(dim=-1)
        loss = criterion(logits.reshape(-1, cfg.num_actions), targets.reshape(-1))
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        sum_loss    += loss.item() * targets.numel()
        sum_correct += (logits.argmax(dim=-1) == targets).sum().item()
        sum_tokens  += targets.numel()
    train_loss = sum_loss / sum_tokens; train_acc = sum_correct / sum_tokens

    model.eval()
    v_loss = v_correct = v_tokens = 0
    with torch.no_grad():
        for rtg, state, action, pos in val_loader:
            rtg = rtg.to(device); state = state.to(device); action = action.to(device)
            pos = pos.to(device) if POS_ENCODING == 'spatial' else None
            logits = model(rtg, state, action, pos_vecs=pos)
            targets = action.argmax(dim=-1)
            loss = criterion(logits.reshape(-1, cfg.num_actions), targets.reshape(-1))
            v_loss    += loss.item() * targets.numel()
            v_correct += (logits.argmax(dim=-1) == targets).sum().item()
            v_tokens  += targets.numel()
    val_loss = v_loss / max(v_tokens, 1); val_acc = v_correct / max(v_tokens, 1)

    history.append({'epoch': epoch, 'train_loss': train_loss, 'train_acc': train_acc,
                    'val_loss': val_loss, 'val_acc': val_acc})
    print(f'[epoch {epoch:3d}] train_loss={train_loss:.4f} acc={train_acc:.3f} '
          f'| val_loss={val_loss:.4f} acc={val_acc:.3f}  ({time.time()-t0:.0f}s)')

ckpt = RUN_DIR / 'model.pt'
torch.save({'state_dict': model.state_dict(), 'cfg': cfg.__dict__, 'arch': 'softmax'}, ckpt)
(RUN_DIR / 'metrics.jsonl').write_text(''.join(json.dumps(h)+'\n' for h in history))
print(f'\nsaved checkpoint -> {ckpt}')

## 4. Inference movie

Roll out the trained DT in `CornerMazeEnv(session_type='exposure')` for visualization. The DT samples actions from `softmax(logits / temperature)`; temperature ramps up if the agent gets stuck (no pose change for a few steps). RTG_TARGET conditions the model: "reward in N steps".

In [ ]:
import imageio.v2 as imageio
from PIL import Image, ImageDraw
from collections import deque

RTG_TARGET   = 20.0    # "reward in ~20 steps"
MAX_STEPS    = 600
BASE_TEMP    = 1.0
ROLLOUT_SEED = 0
MOVIE_PATH   = RUN_DIR / 'rollout.mp4'
ACTION_NAMES = ['Left', 'Right', 'Forward', 'EnterWell', 'Pause']

model.eval()
env = CornerMazeEnv(session_type='exposure', obs_mode='view', render_mode='rgb_array')
env.reset(seed=ROLLOUT_SEED)

buf_state  = deque([np.zeros(EMBED_DIM, dtype=np.float32)] * CONTEXT_SIZE, maxlen=CONTEXT_SIZE)
buf_action = deque([np.zeros(5,         dtype=np.float32)] * CONTEXT_SIZE, maxlen=CONTEXT_SIZE)
rtg_tensor = torch.tensor([[[RTG_TARGET]] * CONTEXT_SIZE], dtype=torch.float32, device=device)

frames = []
last_pose = None; stagnation = 0
total_reward = 0.0; n_well_rewards = 0

for step in range(MAX_STEPS):
    x, y, d = int(env.agent_pos[0]), int(env.agent_pos[1]), int(env.agent_dir)
    if (x, y, d) == last_pose: stagnation += 1
    else: stagnation = 0
    last_pose = (x, y, d)
    temp = BASE_TEMP + (stagnation // 3) * 1.5

    s_vec = encoder.encode(x, y, d)
    buf_state.append(s_vec)
    s_t = torch.from_numpy(np.stack(buf_state)).unsqueeze(0).to(device)
    a_t = torch.from_numpy(np.stack(buf_action)).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(rtg_tensor, s_t, a_t)
        probs = torch.softmax(logits[0, -1, :] / temp, dim=-1)
        action = int(torch.multinomial(probs, num_samples=1).item())

    img = env.render()
    canvas = Image.fromarray(img).resize((416, 416)).convert('RGB')
    panel = Image.new('RGB', (416, 464), 'black')  # 464 divisible by 16 (libx264 macro block)
    panel.paste(canvas, (0, 48))
    draw = ImageDraw.Draw(panel)
    label = (f'step={step:3d} act={ACTION_NAMES[action]} '
             f'rtg={RTG_TARGET:.1f} ret={total_reward:+.2f} wells={n_well_rewards}')
    if stagnation > 0: label += f' stuck={stagnation} T={temp:.1f}'
    draw.text((6, 6),  label,                 fill='white')
    draw.text((6, 24), f'pose=({x},{y},{d})', fill='white')
    frames.append(np.array(panel))

    _, reward, term, trunc, _ = env.step(action)
    total_reward += reward
    if reward > 0.5: n_well_rewards += 1
    oh = np.zeros(5, dtype=np.float32); oh[action] = 1.0
    buf_action.append(oh)
    if term or trunc:
        break

print(f'rollout: {len(frames)} steps, total_reward={total_reward:+.3f}, '
      f'wells={n_well_rewards}')
imageio.mimwrite(MOVIE_PATH, frames, fps=10, codec='libx264')
print(f'wrote -> {MOVIE_PATH}')

## 5. Watch the movie

In [ ]:
from IPython.display import Video
Video(str(MOVIE_PATH), embed=False, width=500)